# 📊 Data Analysis — 10-Week Course
## Week 5: Data Visualisation Principles & Advanced Charts

---
### Learning Objectives
By the end of this week, you will be able to:
- Apply core visualisation design principles (Tufte, pre-attentive attributes)
- Choose the right chart type for each data/question combination
- Build publication-quality figures with Matplotlib and Seaborn
- Create interactive and multi-panel dashboard layouts

### Outline
1. Visualisation Principles
2. Choosing the Right Chart
3. Matplotlib Mastery
4. Seaborn Statistical Charts
5. Multi-Panel Dashboards with GridSpec
6. Annotations & Storytelling
7. Lab Exercises
8. Assignment

---
## 1. Visualisation Principles

### Tufte's Core Rules (adapted)
1. **Show the data** — don't let decoration hide the message
2. **Maximise data-ink ratio** — every pixel should carry information
3. **Erase non-data ink** — remove chart junk (unnecessary gridlines, 3D effects, gap fills)
4. **Avoid distortion** — y-axis should start at 0 for bar charts; dual axes mislead
5. **Serve the story** — design with the audience's question in mind

### Pre-Attentive Attributes (processed in < 250 ms)
| Attribute | Best Used For |
|---|---|
| **Colour (hue)** | Categories |
| **Colour (saturation)** | Magnitude / intensity |
| **Position** | Quantitative comparison |
| **Length / Height** | Magnitude |
| **Size (area)** | Magnitude (use carefully) |
| **Shape** | Category labels |

### Chart Selection Guide
| Question | Chart |
|---|---|
| Distribution of one variable | Histogram, density plot, box plot |
| Compare groups | Bar chart, grouped bar, violin |
| Relationship between two numeric vars | Scatter plot, line plot |
| Part-of-whole | Pie (max 5 slices), stacked bar |
| Change over time | Line chart, area chart |
| Correlation matrix | Heatmap |
| Geographical pattern | Choropleth map |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── Shared style ──────────────────────────────────────────────────────
PALETTE = {
    "West Africa"   : "#3498DB",
    "East Africa"   : "#2ECC71",
    "North Africa"  : "#E74C3C",
    "Central Africa": "#9B59B6",
    "Southern Africa": "#F39C12",
}

plt.rcParams.update({
    "figure.dpi"      : 110,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "axes.grid"       : True,
    "grid.alpha"      : 0.25,
    "font.family"     : "sans-serif",
    "font.size"       : 10,
})

# ── Load dataset ──────────────────────────────────────────────────────
import os
if os.path.exists("africa_health_cleaned.csv"):
    df = pd.read_csv("africa_health_cleaned.csv")
else:
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })

print("Dataset ready:", df.shape)
df.head(3)

---
## 2. Good vs Bad Charts

We'll demonstrate the same data with a poor chart and an improved one.

In [ ]:
region_means = df.groupby("region")["life_expectancy"].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── BAD: 3D-style, rainbow colours, no sorting, truncated axis ────────
ax = axes[0]
bars = ax.bar(region_means.index, region_means.values,
              color=["red","blue","green","orange","purple"],
              edgecolor="black", linewidth=1.5)
ax.set_ylim(55, 75)   # Truncated y-axis exaggerates differences
ax.set_title("❌ Poor Chart Design", fontweight="bold", color="red")
ax.set_ylabel("Life Expectancy")
ax.tick_params(axis="x", rotation=30)
ax.grid(False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.set_xlabel("Issues: truncated axis, rainbow colours, no story")

# ── GOOD: sorted horizontal bar, single colour, zero baseline ─────────
ax = axes[1]
colours = [PALETTE.get(r, "#95A5A6") for r in region_means.index]
ax.barh(region_means.index, region_means.values,
        color=colours, edgecolor="white", alpha=0.9)
ax.set_xlim(0, region_means.max() * 1.12)
ax.set_xlabel("Mean Life Expectancy (years)")
ax.set_title("✓ Improved Chart Design", fontweight="bold", color="green")

# Value labels
for i, (val, region) in enumerate(zip(region_means.values, region_means.index)):
    ax.text(val + 0.3, i, f"{val:.1f} yrs", va="center", fontsize=9)

plt.suptitle("Chart Design: Bad vs Good", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week5_bad_vs_good.png", bbox_inches="tight")
plt.show()

---
## 3. Matplotlib Mastery

### The Object-Oriented Interface
```python
fig, ax = plt.subplots()          # create Figure and Axes
ax.plot(x, y)                      # draw on the Axes
ax.set_xlabel("X label")           # label the axis
ax.set_title("Title")              # set the title
fig.savefig("output.png")          # save the figure
```

### Common Customisations
| Task | Code |
|---|---|
| Change font size | `ax.set_title("T", fontsize=14)` |
| Remove top/right spines | `ax.spines['top'].set_visible(False)` |
| Format tick labels | `ax.xaxis.set_major_formatter(ticker.FuncFormatter(f))` |
| Add text annotation | `ax.annotate("text", xy=(x,y), xytext=(x2,y2), arrowprops={})` |
| Add shaded region | `ax.axvspan(x1, x2, alpha=0.2, color='yellow')` |
| Log scale | `ax.set_xscale('log')` |

In [ ]:
# Demonstrate key Matplotlib customisation features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel A: Styled scatter with annotations ──────────────────────────
ax = axes[0]
for region, grp in df.groupby("region"):
    ax.scatter(np.log(grp["gdp_per_capita"]), grp["life_expectancy"],
               color=PALETTE[region], label=region, alpha=0.75,
               edgecolors="white", s=60)

# Annotate extreme points
top_le = df.nlargest(2, "life_expectancy")
bot_le = df.nsmallest(2, "life_expectancy")
for _, row in pd.concat([top_le, bot_le]).iterrows():
    ax.annotate(
        row["country"],
        xy=(np.log(row["gdp_per_capita"]), row["life_expectancy"]),
        xytext=(8, 4), textcoords="offset points",
        fontsize=7, arrowprops=dict(arrowstyle="->", lw=0.8)
    )

ax.set_xlabel("Log GDP per Capita")
ax.set_ylabel("Life Expectancy (yrs)")
ax.set_title("A — Scatter with Annotations", fontweight="bold")
ax.legend(fontsize=6, loc="lower right")

# ── Panel B: Histogram with custom tick formatter ──────────────────────
ax = axes[1]
n, bins, patches = ax.hist(df["gdp_per_capita"], bins=20,
                            color="#3498DB", edgecolor="white", alpha=0.85)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.axvline(df["gdp_per_capita"].median(), color="red", ls="--",
           lw=1.5, label=f"Median ${df['gdp_per_capita'].median():.0f}")
ax.set_xlabel("GDP per Capita")
ax.set_ylabel("Count")
ax.set_title("B — Custom Tick Format", fontweight="bold")
ax.legend(fontsize=8)

# ── Panel C: Shaded region + twin axes ────────────────────────────────
ax = axes[2]
region_order = df.groupby("region")["life_expectancy"].mean().sort_values().index
means = df.groupby("region")["life_expectancy"].mean()[region_order]
se    = df.groupby("region")["life_expectancy"].sem()[region_order]

x_pos = np.arange(len(means))
ax.bar(x_pos, means.values, color=[PALETTE[r] for r in region_order],
       edgecolor="white", alpha=0.85)
ax.errorbar(x_pos, means.values, yerr=se.values*1.96,
            fmt="none", color="black", capsize=5, lw=1.5)
ax.axhline(df["life_expectancy"].mean(), color="gray", ls=":",
           lw=1.5, label=f"Overall mean {df['life_expectancy'].mean():.1f}")
ax.set_xticks(x_pos)
ax.set_xticklabels(region_order, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Mean Life Expectancy (yrs)")
ax.set_title("C — Bar + CI + Reference Line", fontweight="bold")
ax.legend(fontsize=8)

fig.suptitle("Week 5 — Matplotlib Advanced Features",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week5_matplotlib_features.png", bbox_inches="tight")
plt.show()

---
## 4. Seaborn Statistical Charts

Seaborn builds on Matplotlib to provide:
- Automatic statistical aggregation
- Beautiful default themes
- Figure-level functions (returns a `FacetGrid`)
- Axes-level functions (returns a Matplotlib `Axes`)

| Figure-level | Axes-level equivalent |
|---|---|
| `sns.relplot()` | `sns.scatterplot()`, `sns.lineplot()` |
| `sns.displot()` | `sns.histplot()`, `sns.kdeplot()` |
| `sns.catplot()` | `sns.boxplot()`, `sns.violinplot()`, `sns.barplot()` |

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

region_order_le = df.groupby("region")["life_expectancy"].median().sort_values().index.tolist()
pal = [PALETTE[r] for r in region_order_le]

# A — Box plot
sns.boxplot(data=df, x="region", y="life_expectancy",
            order=region_order_le, palette=PALETTE,
            ax=axes[0, 0])
axes[0, 0].set_title("A — Box Plot", fontweight="bold")
axes[0, 0].tick_params(axis="x", rotation=20)

# B — Violin plot
sns.violinplot(data=df, x="region", y="infant_mortality",
               order=region_order_le, palette=PALETTE,
               inner="quartile", ax=axes[0, 1])
axes[0, 1].set_title("B — Violin Plot", fontweight="bold")
axes[0, 1].tick_params(axis="x", rotation=20)

# C — Swarm / strip
sns.stripplot(data=df, x="region", y="hiv_prevalence",
              order=region_order_le, palette=PALETTE,
              jitter=True, size=6, alpha=0.8, ax=axes[0, 2])
axes[0, 2].set_title("C — Strip Plot", fontweight="bold")
axes[0, 2].tick_params(axis="x", rotation=20)

# D — Bar with error bars
sns.barplot(data=df, x="region", y="vaccination_dtp3",
            order=region_order_le, palette=PALETTE,
            errorbar="sd", capsize=0.1, ax=axes[1, 0])
axes[1, 0].set_title("D — Bar + SD Error Bars", fontweight="bold")
axes[1, 0].tick_params(axis="x", rotation=20)

# E — Regression plot
sns.regplot(data=df, x="water_access", y="life_expectancy",
            scatter_kws={"alpha": 0.6, "color": "#3498DB"},
            line_kws={"color": "#E74C3C", "lw": 2},
            ax=axes[1, 1])
r, p = stats.pearsonr(df["water_access"], df["life_expectancy"])
axes[1, 1].set_title(f"E — Regplot (r={r:.2f}, p={p:.3f})", fontweight="bold")

# F — KDE plot
for region in df["region"].unique():
    subset = df[df["region"] == region]["life_expectancy"]
    sns.kdeplot(subset, label=region, color=PALETTE[region],
                fill=True, alpha=0.15, ax=axes[1, 2])
axes[1, 2].set_title("F — KDE by Region", fontweight="bold")
axes[1, 2].legend(fontsize=7)

fig.suptitle("Week 5 — Seaborn Statistical Charts",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week5_seaborn_showcase.png", bbox_inches="tight")
plt.show()

---
## 5. Multi-Panel Dashboards with GridSpec

`GridSpec` gives you full control over panel sizes and spans — ideal for combining charts of different sizes into a polished dashboard layout.

```python
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.3)

ax_big   = fig.add_subplot(gs[0:2, 0:3])  # spans 2 rows, 3 cols
ax_right = fig.add_subplot(gs[0:2, 3])    # spans 2 rows, 1 col
ax_bot1  = fig.add_subplot(gs[2, 0:2])    # bottom-left 2 cols
ax_bot2  = fig.add_subplot(gs[2, 2:])     # bottom-right 2 cols
```

In [ ]:
# Full Africa Health Dashboard using GridSpec
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor("#F8F9FA")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel A (large): Scatter GDP vs LE ────────────────────────────────
ax_a = fig.add_subplot(gs[0:2, 0:2])
for region, grp in df.groupby("region"):
    ax_a.scatter(np.log(grp["gdp_per_capita"]), grp["life_expectancy"],
                 color=PALETTE[region], label=region, alpha=0.8,
                 edgecolors="white", s=70)
ax_a.set_xlabel("Log GDP per Capita")
ax_a.set_ylabel("Life Expectancy (yrs)")
ax_a.set_title("A — GDP vs Life Expectancy", fontweight="bold")
ax_a.legend(fontsize=7, loc="lower right")
ax_a.set_facecolor("white")

# ── Panel B: Violin — LE by region ────────────────────────────────────
ax_b = fig.add_subplot(gs[0:2, 2:4])
region_order_le = df.groupby("region")["life_expectancy"].median().sort_values().index
sns.violinplot(data=df, x="region", y="life_expectancy",
               order=region_order_le, palette=PALETTE,
               inner="box", alpha=0.8, ax=ax_b)
ax_b.set_title("B — Life Expectancy by Region", fontweight="bold")
ax_b.set_xlabel("")
ax_b.tick_params(axis="x", rotation=20)
ax_b.set_facecolor("white")

# ── Panel C: Correlation heatmap ──────────────────────────────────────
ax_c = fig.add_subplot(gs[2, 0:2])
num_cols = df.select_dtypes("number").columns
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".1f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar=False,
            ax=ax_c, annot_kws={"size": 7})
ax_c.set_title("C — Correlation Matrix", fontweight="bold")
ax_c.tick_params(axis="x", rotation=45, labelsize=7)
ax_c.tick_params(axis="y", labelsize=7)

# ── Panel D: Top 10 countries by LE ──────────────────────────────────
ax_d = fig.add_subplot(gs[2, 2:4])
top10 = df.nlargest(10, "life_expectancy")[["country","life_expectancy","region"]]
colors_d = [PALETTE[r] for r in top10["region"]]
ax_d.barh(top10["country"], top10["life_expectancy"],
          color=colors_d, edgecolor="white", alpha=0.9)
ax_d.set_xlabel("Life Expectancy (yrs)")
ax_d.set_title("D — Top 10 Countries", fontweight="bold")
ax_d.set_xlim(0, top10["life_expectancy"].max() * 1.1)
for i, (val, _) in enumerate(zip(top10["life_expectancy"], top10["country"])):
    ax_d.text(val + 0.2, i, f"{val:.1f}", va="center", fontsize=8)

# Legend patches
legend_patches = [mpatches.Patch(color=c, label=r) for r, c in PALETTE.items()]
fig.legend(handles=legend_patches, loc="lower center",
           ncol=5, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, -0.01))

fig.suptitle("Africa Health Dashboard — Week 5",
             fontsize=16, fontweight="bold", y=1.01)
fig.savefig("week5_dashboard.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Dashboard saved → week5_dashboard.png")

---
## 6. Annotations & Storytelling

A chart becomes a **story** when you highlight the key message directly on the figure.

### Annotation Toolkit
| Element | Code |
|---|---|
| Text label | `ax.text(x, y, "text", fontsize=10)` |
| Arrow annotation | `ax.annotate("text", xy=point, xytext=offset, arrowprops={})` |
| Horizontal/vertical line | `ax.axhline(val)` / `ax.axvline(val)` |
| Shaded region | `ax.axvspan(x1, x2, alpha=0.2)` |
| Text box | `ax.text(..., bbox=dict(boxstyle='round', facecolor='wheat'))` |

In [ ]:
# Storytelling chart: highlight countries that need most attention
fig, ax = plt.subplots(figsize=(11, 7))

# Background: all countries
ax.scatter(df["water_access"], df["life_expectancy"],
           color="#BDC3C7", alpha=0.5, s=60, zorder=2, label="Other countries")

# Critical quadrant: low water & low LE
thresh_water = 50
thresh_le    = 60
critical = df[(df["water_access"] < thresh_water) & (df["life_expectancy"] < thresh_le)]
ax.scatter(critical["water_access"], critical["life_expectancy"],
           color="#E74C3C", s=100, zorder=3, edgecolors="white",
           label=f"High-risk countries (n={len(critical)})")

# Annotate top 5 most critical
for _, row in critical.nsmallest(5, "life_expectancy").iterrows():
    ax.annotate(row["country"],
                xy=(row["water_access"], row["life_expectancy"]),
                xytext=(8, 4), textcoords="offset points",
                fontsize=8, color="#C0392B",
                arrowprops=dict(arrowstyle="->", color="#C0392B", lw=0.8))

# Reference lines
ax.axvline(thresh_water, color="#E74C3C", ls="--", lw=1.2, alpha=0.6)
ax.axhline(thresh_le,    color="#E74C3C", ls="--", lw=1.2, alpha=0.6)

# Quadrant shading
ax.axvspan(ax.get_xlim()[0] if ax.get_xlim()[0] else 0,
           thresh_water, ymax=0.5, color="#FADBD8", alpha=0.3, zorder=1)

# Story annotation
ax.text(18, 73,
        "Countries in the red zone have\nboth low water access (<50%)\n"
        "AND low life expectancy (<60 yrs).\nThey need priority attention.",
        fontsize=9, color="#C0392B",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#FADBD8", edgecolor="#E74C3C"))

# Regression line
m, b = np.polyfit(df["water_access"], df["life_expectancy"], 1)
x_fit = np.linspace(df["water_access"].min(), df["water_access"].max(), 100)
ax.plot(x_fit, m*x_fit + b, color="#2C3E50", lw=1.5, ls="-.", label="Trend")

ax.set_xlabel("Population with Access to Clean Water (%)", fontsize=11)
ax.set_ylabel("Life Expectancy (years)", fontsize=11)
ax.set_title("Water Access vs Life Expectancy\nHighlighting High-Risk Countries",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("week5_storytelling.png", bbox_inches="tight")
plt.show()

---
## 7. Lab Exercises

### Lab 1: Redesign a Chart
The code below produces a poor chart. Improve it: sort bars, remove non-data ink, add value labels, and use a single meaningful colour.

In [ ]:
# Lab 1: Poor chart to redesign
region_vax = df.groupby("region")["vaccination_dtp3"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Poor version (provided)
axes[0].bar(region_vax.index, region_vax.values,
            color=["red","blue","green","yellow","purple"])
axes[0].set_title("Vaccination Rate by Region (ORIGINAL)")

# TODO: Improved version in axes[1]
# 1. Sort the bars
# 2. Use a single colour (e.g. #3498DB)
# 3. Add value labels at the end of each bar
# 4. Remove top/right spines
# 5. Add a descriptive title

plt.tight_layout()
plt.show()

### Lab 2: FacetGrid Analysis
Create a `FacetGrid` showing the distribution of `infant_mortality` for each region as a KDE plot. Add a vertical line for each region's median.

In [ ]:
# Lab 2
# TODO: FacetGrid — infant_mortality KDE per region + median line
pass

### Lab 3: Mini Dashboard
Using `GridSpec`, create a 2×2 dashboard with:
- Top-left: scatter `health_expenditure` vs `life_expectancy`
- Top-right: violin `maternal_mortality` by region
- Bottom-left: horizontal bar of top 8 vaccination countries
- Bottom-right: KDE of `life_expectancy` (overall)

In [ ]:
# Lab 3
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
# TODO: Add 4 panels
plt.show()

---
## 8. Assignment — Week 5

**Problem 1 (20 pts):** Create a "before and after" visualisation demonstrating the difference between a truncated y-axis bar chart and a correctly scaled one. Use any variable from the Africa Health dataset.

**Problem 2 (30 pts):** Build a `3 × 2` GridSpec dashboard that tells the story **"Which African regions have the best overall health outcomes?"**. Include at least:
- One comparison chart (box/violin)
- One scatter/bubble chart
- One ranking chart (sorted bar)
- Descriptive titles and value labels where appropriate

**Problem 3 (25 pts):** Create an **annotated storytelling chart** that highlights countries where `hiv_prevalence > 10%`. Shade the high-risk zone, annotate the three highest-HIV countries, and add a 2-sentence insight box.

**Problem 4 (25 pts):** Write a 150-word critique of a chart you find online (or in a textbook). Apply Tufte's principles to identify two strengths and two weaknesses, and describe one specific improvement.

In [ ]:
# Problem 1
pass  # TODO

In [ ]:
# Problem 2 — GridSpec dashboard
pass  # TODO

In [ ]:
# Problem 3 — Storytelling chart: HIV prevalence
pass  # TODO

**Problem 4 — Written chart critique (150 words):**

> *Type your answer here.*

---
## Summary — Week 5

| Concept | Key Point |
|---|---|
| Tufte's principles | Maximise data-ink; remove chart junk; no truncated axes |
| Pre-attentive attributes | Colour, position, length processed in < 250 ms |
| Chart selection | Match chart type to data type and question |
| Matplotlib OO | `fig, ax = plt.subplots()` → full control over every element |
| Seaborn | Statistical charts with less code; integrates with Pandas |
| GridSpec | Flexible multi-panel layouts with variable sizes |
| Storytelling | Annotations + shading + focused highlighting drive the message |

**Next:** Week 6 — Statistical Analysis & Hypothesis Testing